# Vígil.ia — Servidor de demo na GPU do Colab (RT-DETR)

Transforma esta sessão do Colab no **backend de visão do Vígil.ia**: mesmo contrato
da API `/inspect` do HF Space, mas rodando o **RT-DETR ft_v3 na GPU** (~70 fps)
com NMS agnóstico de classe.

**Como usar no dia da demo:**
1. Rodar todas as células (~3 min). A última imprime um link `https://xxxx.trycloudflare.com`.
2. Abrir o site com `?api=<esse link>` no final, ex.:
   `https://SEU-SITE.vercel.app/?api=https://xxxx.trycloudflare.com`
3. O site inteiro passa a usar a GPU do Colab. Pra voltar ao normal: `?api=off`.

⚠️ Abrir a sessão ~30 min ANTES da apresentação (não de véspera — Colab derruba
sessão ociosa). O link muda a cada sessão.

## 1. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80" fastapi uvicorn

import torch
assert torch.cuda.is_available(), 'Sem GPU! Ambiente de execução -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Modelo (RT-DETR ft_v3 do Drive) + patch NMS + aquecimento

In [ ]:
import numpy as np
import torchvision
from google.colab import drive
from ultralytics import RTDETR
from ultralytics.models.rtdetr.predict import RTDETRPredictor

drive.mount('/content/drive')
MODEL_PT = '/content/drive/MyDrive/soja_rtdetr_ft_v3.pt'
CONF = 0.35

# NMS agnóstico de classe (RT-DETR não roda NMS por padrão -> caixa dupla)
if not hasattr(RTDETRPredictor, '_pp_orig'):
    RTDETRPredictor._pp_orig = RTDETRPredictor.postprocess

def _pp_nms(self, preds, img, orig_imgs):
    results = RTDETRPredictor._pp_orig(self, preds, img, orig_imgs)
    for r in results:
        if len(r.boxes) > 1:
            keep = torchvision.ops.nms(r.boxes.xyxy, r.boxes.conf, iou_threshold=0.6)
            r.update(boxes=r.boxes.data[keep])
    return results

RTDETRPredictor.postprocess = _pp_nms

model = RTDETR(MODEL_PT)
NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']

# aquecimento (compila kernels; 1ª inferência é lenta)
_ = model.predict(np.zeros((640, 640, 3), dtype=np.uint8), conf=CONF,
                  imgsz=640, device=0, verbose=False)
print('modelo pronto ✅ (patch NMS aplicado, GPU aquecida)')

## 3. API — mesmo contrato do backend do Vígil.ia
`GET /health` e `POST /inspect` idênticos ao HF Space (o front nem percebe a troca).
Sem banco: cada inspeção ganha um uuid e o front usa o sessionStorage, como sempre.

In [ ]:
import base64, io, threading, uuid

import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from pydantic import BaseModel

app = FastAPI(title='Vígil.ia demo GPU (RT-DETR)')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=False,
                   allow_methods=['*'], allow_headers=['*'])

class InspectRequest(BaseModel):
    image: str
    imagem_url: str = ''
    multi: bool = False   # ignorado: RT-DETR é multi-grão nativo
    persist: bool = True  # sem banco na demo

@app.get('/health')
def health():
    return {'status': 'ok', 'engine': 'rtdetr-ft_v3-gpu'}

@app.post('/inspect')
def inspect(body: InspectRequest):
    try:
        b64 = body.image.split(',', 1)[1] if ',' in body.image else body.image
        img = Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB')
    except Exception as exc:
        raise HTTPException(status_code=422, detail=f'Imagem inválida: {exc}')

    try:
        r = model.predict(img, conf=CONF, imgsz=640, device=0, verbose=False)[0]
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f'Erro de inferência: {exc}')

    detections, counts = [], {}
    for xyxy, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                           r.boxes.cls.int().tolist(),
                           r.boxes.conf.tolist()):
        name = NAMES[c]
        detections.append({'class': name, 'class_id': c,
                           'confidence': round(cf, 4),
                           'bbox': [int(v) for v in xyxy]})
        counts[name] = counts.get(name, 0) + 1

    w, h = img.size
    return {'id': str(uuid.uuid4()), 'total_graos': len(detections),
            'class_counts': counts, 'detections': detections,
            'image_width': w, 'image_height': h}

server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()

# auto-teste local
import time, requests
time.sleep(2)
print('health:', requests.get('http://127.0.0.1:8000/health').json())
buf = io.BytesIO()
Image.new('RGB', (64, 64), (30, 30, 30)).save(buf, format='JPEG')
probe = 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()
t0 = time.perf_counter()
resp = requests.post('http://127.0.0.1:8000/inspect', json={'image': probe})
print(f'inspect: HTTP {resp.status_code} em {(time.perf_counter()-t0)*1000:.0f} ms ✅')

## 4. Túnel público (Cloudflare, grátis, sem conta) → LINK DA DEMO

In [ ]:
import re, subprocess, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
t0 = time.time()
while time.time() - t0 < 30:
    line = proc.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
    if m:
        url = m.group(0)
        break
assert url, 'Túnel não subiu — rode a célula de novo.'

import requests
for _ in range(10):  # espera o túnel propagar
    try:
        if requests.get(f'{url}/health', timeout=5).ok:
            break
    except Exception:
        time.sleep(2)

print('=' * 64)
print('LINK DO BACKEND GPU:', url)
print('=' * 64)
print('\nAbra o site assim (troque SEU-SITE pelo domínio da Vercel):')
print(f'  https://SEU-SITE.vercel.app/?api={url}')
print('\nPra voltar ao backend normal depois: .../?api=off')
print('Deixe esta célula/sessão RODANDO durante toda a demo.')

## Dicas de dia de demo

- Rode tudo **~30 min antes** da apresentação e faça 1 inspeção de teste pelo site.
- Se o link cair (sessão reciclada): re-rode a célula 4 → sai um link novo → atualiza o `?api=`.
- A 1ª inspeção pode levar ~2 s (aquecimento do túnel); as seguintes ficam < 1 s.
- Fallback à prova de falha: `?api=off` volta pro HF Space (modo foto que já funciona).